# FBref

In [ ]:
! pip install undetected-chromedriver beautifulsoup4 pandas mysql-connector-python thefuzz

## Ids times

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def extrair_ids_com_genero():
    """
    Extrai lista de IDs (identificadores únicos) de todos os clubes brasileiros
    do FBRef, filtrando apenas times masculinos
    Utiliza Selenium com driver desprotegido para bypass de Cloudflare
    """
    # ====== INICIALIZAR DRIVER ======
    # Cria instância do Chrome com recursos anti-detecção ativados
    # Bypass automático para proteção Cloudflare
    # version_main=146: Versão estável do Chrome compatível com FBRef (2026)
    options = uc.ChromeOptions()
    driver = uc.Chrome(options=options, version_main=146) 
    
    # URL da página com lista de clubes brasileiros
    url = "https://fbref.com/en/country/clubs/BRA/Brazil-Football-Clubs"
    
    try:
        print(f"🚀 Acessando FBRef (página de clubes brasileiros)...")
        driver.get(url)
        
        # ====== AGUARDAR CARREGAMENTO ======
        # time.sleep(15): Delay crítico para:
        # 1. Resolução de desafio Cloudflare (se necessário)
        # 2. Renderização completa do conteúdo JavaScript
        # 3. Carregamento de tabelas dinâmicas
        time.sleep(15) 
        
        # ====== PARSEAR CONTEÚDO ======
        # Extrai HTML já renderizado (inclui conteúdo dinâmico carregado por JS)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # ====== BUSCAR TABELA DE TIMES ======
        # Percorre todas as linhas de tabela para extrair dados
        rows = soup.find_all('tr')
        
        clubes = []
        for row in rows:
            # ====== EXTRAIR LINK DO TIME ======
            # Busca por link (tag <a>) que corresponde ao padrão de URLs de squads do FBRef
            # Padrão: /en/squads/{8-char-id}/
            link = row.find('a', href=re.compile(r'/en/squads/[a-f0-9]{8}/'))
            
            # ====== EXTRAIR GÊNERO ======
            # Busca célula com atributo data-stat='gender' (M=Masculino, W=Feminino)
            gender_cell = row.find('td', {'data-stat': 'gender'})
            
            # ====== VALIDAR LINHA ======
            # Verifica se a linha contém os dados esperados
            if link and gender_cell:
                # Extrai o ID canônico do FBRef a partir da URL
                # Formato: https://fbref.com/en/squads/[ESTE_ID]/...
                href = link['href']
                # href.split('/')[3]: Índice 3 na divisão por '/' retorna o ID de 8 caracteres
                # Exemplo: "/en/squads/abc12345/..." → ["", "en", "squads", "abc12345", ...]
                time_id = href.split('/')[3]
                
                # Extrai nome do time e gênero
                nome_time = link.get_text(strip=True)
                genero = gender_cell.get_text(strip=True)
                
                # ====== APLICAR FILTRO ======
                # Regra de negócio: filtrar apenas times masculinos
                # Descarta times femininos (W) para análise específica
                if genero == "M":
                    clubes.append({
                        'nome_fbref': nome_time,      # Nome canônico do FBRef
                        'fbref_id': time_id,          # ID único de 8 caracteres
                        'genero': genero              # Confirmação de gênero (M)
                    })

        # ====== DEDUPLICAÇÃO ======
        # Remove registros duplicados por ID (defensivo contra inconsistências do HTML)
        # Um ID deve aparecer apenas uma vez
        df = pd.DataFrame(clubes).drop_duplicates(subset=['fbref_id'])
        
        return df

    finally:
        # ====== LIMPEZA DE RECURSOS ======
        # Encerra o driver em qualquer caso (sucesso ou erro)
        # Garante liberação do processo Chrome (evita zumbis)
        driver.quit()


# ====== EXECUTAR EXTRAÇÃO ======
# Chamada isolada da função de scraping
df_ids_limpo = extrair_ids_com_genero()

# ====== PERSISTÊNCIA EM CSV ======
# Salva resultado em formato padrão de dados tabulares
# UTF-8-sig adiciona BOM (Byte Order Mark) para compatibilidade com Excel
df_ids_limpo.to_csv(
    'csv//ids_times_fbref_masculino.csv',
    index=False,      # Não salva índice sequencial do pandas
    sep=';',          # Separador de coluna padrão brasileiro
    encoding='utf-8-sig'  # Compatível com Excel (inclui BOM)
)

# ====== FEEDBACK DE EXECUÇÃO ======
# Informa resultado do processamento ao usuário
print(f"✅ Extração concluída with sucesso. {len(df_ids_limpo)} times masculinos encontrados e salvos em CSV.")

In [ ]:
df_ids_limpo[df_ids_limpo['fbref_id'] == '7a1064a2']

## Mapeamento times

In [ ]:
import pandas as pd
import re
from thefuzz import fuzz, process

# ====== STEP 1: Carregar dados de entrada ======
# Carrega os dois arquivos CSV necessários para o mapeamento
df_meus_times = pd.read_csv('csv//times_2025.csv', sep=';')
df_fbref = pd.read_csv('csv//ids_times_fbref_masculino.csv', sep=';')

# Limpeza inicial: remove espaços em branco para evitar falhas no matching
df_meus_times['nome'] = df_meus_times['nome'].str.strip()
df_fbref['nome_fbref'] = df_fbref['nome_fbref'].str.strip()

def limpar_nome_time(nome):
    """
    Normaliza nomes de times removendo sufixos comuns (SC, FC, Clube, etc)
    para melhorar o fuzzy matching entre bases com nomenclaturas diferentes
    """
    if not nome: return ""
    nome = nome.upper()
    # Lista de sufixos e prefixos que podem variar entre bases de dados
    termos = [r'\bSC\b', r'\bFC\b', r'\bEC\b', r'\bSAF\b', r'\bFR\b', 
              r'\bCLUBE\b', r'\bESPORTE\b', r'\bFUTEBOL\b', r'\bASSOCIACAO\b']
    for termo in termos:
        nome = re.sub(termo, '', nome)
    return nome.strip()

# Prepara lista de nomes FBRef para uso no matching de cada time
lista_fbref = df_fbref['nome_fbref'].tolist()

def mapear_times_robusto(nome_meu_banco, escolhas_fbref):
    """
    Mapeia nomes de times entre bases com 3 estratégias em cascata:
    1. Match exato (mais confiável, retorna imediatamente)
    2. Fuzzy matching com score >= 65% (varia nomenclatura)
    3. Fuzzy matching após limpeza com score >= 80% (remove sufixos)
    Retorna None se nenhuma estratégia conseguir match
    """
    # ESTRATÉGIA 1: Busca por match exato (rápido e seguro)
    if nome_meu_banco in escolhas_fbref:
        return nome_meu_banco
    
    # ESTRATÉGIA 2: Fuzzy matching direto com threshold de 65%
    # resultado[1]: Score de 0-100 indicando similaridade percentual
    # Threshold 65%: Considera matches com 65% de semelhança (ex: "SC Corinthians" vs "Corinthians")
    resultado = process.extractOne(nome_meu_banco, escolhas_fbref, scorer=fuzz.token_sort_ratio)
    
    if resultado and resultado[1] >= 65:
        return resultado[0]
    
    # ESTRATÉGIA 3: Fuzzy matching após limpeza de sufixos (score >= 80%)
    # Aumenta threshold para 80% para maior confiabilidade após normalização
    # Casos: remove "SC", "FC", etc. para comparar apenas o core do nome
    nome_limpo = limpar_nome_time(nome_meu_banco)
    escolhas_limpas = [limpar_nome_time(x) for x in escolhas_fbref]
    
    resultado_limpo = process.extractOne(nome_limpo, escolhas_limpas, scorer=fuzz.token_sort_ratio)
    
    if resultado_limpo and resultado_limpo[1] >= 80:
        # Mantém o nome original do FBRef encontrando seu índice na lista limpa
        idx = escolhas_limpas.index(resultado_limpo[0])
        return escolhas_fbref[idx]

    # Nenhuma estratégia funcionou - retorna None para análise posterior
    return None

print("🔄 Iniciando mapeamento robusto de times...")

# ====== STEP 2: Executar matching entre bases de dados ======
# Aplica a função de mapeamento a cada time da base local
df_meus_times['nome_time_fbref'] = df_meus_times['nome'].apply(lambda x: mapear_times_robusto(x, lista_fbref))

# ====== STEP 3: Juntar os dados com os IDs FBRef ======
# Merge left para manter todos os times da base local (mesmo sem match no FBRef)
df_final = pd.merge(
    df_meus_times, 
    df_fbref[['nome_fbref', 'fbref_id']], 
    left_on='nome_time_fbref', 
    right_on='nome_fbref', 
    how='left'
)

# ====== STEP 4: Padronizar estrutura de saída ======
# Seleciona e renomeia colunas para o padrão esperado
df_final = df_final[['nome', 'nome_time_fbref', 'fbref_id']]
df_final.columns = ['nome_time', 'nome_time_fbref', 'id_fbref']

# ====== STEP 5: Persistir resultado ======
# Salva o mapeamento consolidado em CSV para uso em extrações posteriores
df_final.to_csv('csv//mapeamento_times.csv', index=False, sep=';', encoding='utf-8-sig')

print("✅ Mapeamento concluído com sucesso!")
print(f"Total de times mapeados: {len(df_final)}")
print(df_final.head(15))

In [ ]:
df_final.shape

In [ ]:
df_final.isna().sum()

## Extração de Jogadores (2025)

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re

# Carrega mapeamento previamente gerado (fonte de verdade para IDs do FBref)
df_mapeamento = pd.read_csv('csv//mapeamento_times.csv', sep=';')

# Remove registros inválidos (IDs ausentes inviabilizam construção da URL)
df_mapeamento = df_mapeamento.dropna(subset=['id_fbref'])


def extrair_jogadores_2025():
    """
    Extrai lista de jogadores de cada time da temporada 2025
    Scraping via Selenium com bypass anti-bot para Cloudflare
    """
    # Inicializa driver com bypass de proteção anti-bot
    # version_main=146: Versão estável do Chrome compatível com FBRef (2026)
    options = uc.ChromeOptions()
    # options.add_argument('--headless')  # Evitar headless em fase inicial (Cloudflare mais restritivo)
    
    driver = uc.Chrome(options=options, version_main=146)
    
    dados_totais = []

    try:
        # Iteração sequencial através de todos os times mapeados
        # Necessária para respeitar rate limit do FBref e evitar bloqueios
        for index, row in df_mapeamento.iterrows():
            nome_original = row['nome_time']
            id_fbref = row['id_fbref']
            
            # Construção determinística da URL específica para 2025
            url_2025 = f"https://fbref.com/en/squads/{id_fbref}/2025/Stats"
            
            print(f"⏳ ({index+1}/{len(df_mapeamento)}) Acessando 2025 para: {nome_original}...")
            
            driver.get(url_2025)
            
            # time.sleep(15): Delay crítico para evitar bloqueio (HTTP 429 / challenge Cloudflare)
            # 15 segundos garante renderização completa do conteúdo JS + resolução de challenge
            time.sleep(15) 
            
            # Parse do DOM completo após renderização JavaScript
            soup = BeautifulSoup(driver.page_source, "html.parser")
            
            # Busca resiliente pela tabela de estatísticas padrão
            # ID varia por time (ex: stats_standard_9) então usa regex para encontrar
            tabela_jogadores = soup.find('table', {
                'id': re.compile(r'stats_standard_')
            })
            
            if tabela_jogadores:
                tbody = tabela_jogadores.find('tbody')
                
                # Proteção contra HTML inconsistente ou incompletamente carregado
                if not tbody:
                    print(f"⚠️ {nome_original}: tabela sem tbody (conteúdo incompleto)")
                    continue
                
                rows = tbody.find_all('tr')
                
                # Itera por cada linha da tabela de jogadores
                for r in rows:
                    # Coluna de jogador fica no elemento <th> com data-stat='player' (padrão FBref)
                    col_jogador = r.find('th', {'data-stat': 'player'})
                    
                    if col_jogador:
                        nome_jogador = col_jogador.get_text(strip=True)
                        
                        # Nem todos os jogadores possuem link (ex: linhas de agregação)
                        link_tag = col_jogador.find('a')
                        link_jogador = link_tag['href'] if link_tag else ''
                        
                        # Estrutura de dados pronta para salvamento
                        dados_totais.append({
                            'time_banco': nome_original,
                            'id_fbref_time': id_fbref,
                            'nome_jogador': nome_jogador,
                            'url_fbref_jogador': f"https://fbref.com{link_jogador}" if link_jogador else '',
                            'temporada': '2025'
                        })
                
                print(f"✅ {nome_original}: jogadores extraídos com sucesso")
            
            else:
                # Possíveis causas de falha:
                # 1. Time não possui quadro completo publicado em 2025 ainda
                # 2. Requisição foi bloqueada (HTTP 429, Cloudflare challenge)
                # 3. Layout do FBRef foi alterado (mudança de estrutura HTML)
                # 4. Conteúdo não foi renderizado corretamente pelo JS
                print(f"❌ {nome_original}: tabela de 2025 não encontrada (verifique logs acima)")

        # Retorna DataFrame consolidado pronto para persistência em CSV
        return pd.DataFrame(dados_totais)

    finally:
        # Garante encerramento do driver mesmo em caso de erro
        # Evita processos "zumbis" (orphaned processes) do Chrome
        driver.quit()


# Execução isolada da função de extração
df_jogadores_2025 = extrair_jogadores_2025()

# Persistência apenas se houver dados (evita gerar arquivo vazio)
if not df_jogadores_2025.empty:
    df_jogadores_2025.to_csv(
        'csv//jogadores_extraidos_2025.csv',
        index=False,
        sep=';',
        encoding='utf-8-sig'  # Compatível com Excel (inclui BOM)
    )
    
    print(f"\n🚀 Finalizado! Total de {len(df_jogadores_2025)} jogadores salvos em CSV.")

## Estatísticas Completas (2025)

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

# Carregar mapeamento de times com IDs do FBref (fonte central de verdade)
df_map = pd.read_csv('csv//mapeamento_times.csv', sep=';').dropna(subset=['id_fbref'])


def traduzir_posicao(pos_en):
    """
    Converte códigos de posição do FBRef (em inglês) para português do Brasil
    Exemplo: GK → Goleiro, CB,LB → Zagueiro, Lateral Esquerdo
    """
    if not pos_en:
        return "N/I"  # Não informado
    
    # Mapeamento completo de traduções
    mapa = {
        'GK': 'Goleiro',
        'DF': 'Defensor',
        'MF': 'Meio-campista',
        'FW': 'Atacante',
        'FB': 'Lateral',
        'LB': 'Lateral Esquerdo',
        'RB': 'Lateral Direito',
        'CB': 'Zagueiro'
    }
    
    # Divide por vírgula (jogadores com múltiplas posições), traduz cada uma, junta de novo
    partes = pos_en.split(',')
    partes_traduzidas = [mapa.get(p.strip(), p.strip()) for p in partes]
    
    return ', '.join(partes_traduzidas)

def extrair_stats_completas_portugues_2025():
    """
    Extrai todas as estatísticas de jogadores para 2025, incluindo:
    - Dados pessoais (nome, nacionalidade, posição, idade)
    - Minutagem (jogos, titularidade, minutos absolutos e normalizados)
    - Performance (gols, assistências, cartões, etc)
    Traduz labels para português do Brasil
    """
    # Inicializa driver anti-bot com suporte a Cloudflare
    # version_main=146: Versão estável do Chrome compatível com FBRef (2026)
    options = uc.ChromeOptions()
    # options.add_argument('--headless') 
    driver = uc.Chrome(options=options, version_main=146)
    
    lista_jogadores = []

    try:
        # Itera por cada time no mapeamento consolidado
        for index, row in df_map.iterrows():
            time_nome = row['nome_time']
            fbref_id = row['id_fbref']
            
            # Construção determinística da URL específica para 2025
            url = f"https://fbref.com/en/squads/{fbref_id}/2025/Stats"
            
            print(f"📊 ({index+1}/{len(df_map)}) Extraindo: {time_nome}...")
            
            driver.get(url)
            # time.sleep(20): Aguarda 20 segundos para garantir carregamento JS e evitar HTTP 429
            # Página de stats é mais pesada (múltiplas tabelas + formulários) que lista de jogadores
            # Este delay maior é crítico pois o FBRef penaliza requisições muito frequentes
            time.sleep(20) 
            
            soup = BeautifulSoup(driver.page_source, "html.parser")
            
            # Localiza a tabela "stats_standard" (tabela principal de estatísticas)
            # Usa regex pois o ID tem sufixo numérico variável por time
            table = soup.find('table', {'id': re.compile(r'stats_standard_')})
            
            if not table:
                print(f"⚠️ Dados de 2025 ainda não disponíveis para {time_nome}.")
                continue

            rows = table.find('tbody').find_all('tr')
            
            # Processa cada linha da tabela de estatísticas
            for r in rows:
                # Ignora linhas de separação/espaçamento (class='spacer')
                if r.get('class') and 'spacer' in r.get('class'):
                    continue
                
                # Função auxiliar para extrair valores de células pelo atributo data-stat
                def get_val(stat_name):
                    cell = r.find(['td', 'th'], {'data-stat': stat_name})
                    return cell.get_text(strip=True) if cell else "0"

                # Extrai todas as estatísticas disponíveis e traduz labels para português
                stats = {
                    'time': time_nome,
                    'nome': get_val('player'),
                    'nacionalidade': get_val('nationality'),
                    'posicao': traduzir_posicao(get_val('position')),
                    'idade': get_val('age'),
                    'qt_jogos': get_val('games'),
                    'qt_jogos_titular': get_val('games_starts'),
                    'minutos_jogados': get_val('minutes'),                           # Minutos absolutos
                    'minutos_jogados_divid_90': get_val('minutes_90s'),              # Minutos normalizados
                    'gols': get_val('goals'),
                    'assistencias': get_val('assists'),
                    'participacoes_gols': get_val('goals_assists'),                 # Gols + Assistências
                    'gols_nao_penalti': get_val('goals_pens'),
                    'gols_penalti': get_val('pens_made'),
                    'chutes_penalti': get_val('pens_att'),
                    'cartoes_amarelo': get_val('cards_yellow'),
                    'cartoes_vermelho': get_val('cards_red'),
                    'gols_90': get_val('goals_per90'),                              # Gols por 90 minutos
                    'assistencias_90': get_val('assists_per90'),                    # Assistências por 90 minutos
                    'participacoes_gols_90': get_val('goals_assists_per90'),        # Combo por 90 minutos
                    'gols_nao_penalti_90': get_val('goals_pens_per90'),
                    'participacoes_gols_e_penalti_90': get_val('goals_assists_pens_per90')
                }
                
                # Valida se o registro representa um jogador real (tem nome)
                if stats['nome']:
                    lista_jogadores.append(stats)

            print(f"✅ {time_nome} processado com sucesso.")

        # Retorna DataFrame consolidado com todas as estatísticas coletadas
        return pd.DataFrame(lista_jogadores)

    finally:
        # Garante encerramento do driver mesmo em caso de erro
        driver.quit()

# ====== EXECUÇÃO DO SCRAPING ======
df_stats_traduzido = extrair_stats_completas_portugues_2025()

# ====== PERSISTÊNCIA EM CSV ======
if not df_stats_traduzido.empty:
    # Cria diretório CSV se não existir
    if not os.path.exists('csv'): os.makedirs('csv')
    
    # Salva resultado consolidado
    output_path = 'csv//stats_jogadores_2025.csv'
    df_stats_traduzido.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
    
    print(f"\n🚀 Pronto! Arquivo salvo em: {output_path}")
    print(f"Total de registros: {len(df_stats_traduzido)}")
    print(f"Colunas geradas: {list(df_stats_traduzido.columns)}")
else:
    print("\n❌ Nenhuma estatística coletada. Verifique os logs acima para identificar o problema.")

📊 (2/31) Extraindo: Botafogo-SP...
✅ Botafogo-SP processado com sucesso.
📊 (3/31) Extraindo: Corinthians...
✅ Corinthians processado com sucesso.
📊 (4/31) Extraindo: Guarani...
⚠️ Dados de 2025 ainda não disponíveis para Guarani.
📊 (6/31) Extraindo: Mirassol...
✅ Mirassol processado com sucesso.
📊 (8/31) Extraindo: Novorizontino...
✅ Novorizontino processado com sucesso.
📊 (9/31) Extraindo: Palmeiras...
✅ Palmeiras processado com sucesso.
📊 (12/31) Extraindo: RB Bragantino...
✅ RB Bragantino processado com sucesso.
📊 (13/31) Extraindo: Santos...
✅ Santos processado com sucesso.
📊 (14/31) Extraindo: São Bernardo...
⚠️ Dados de 2025 ainda não disponíveis para São Bernardo.
📊 (15/31) Extraindo: São Paulo...
✅ São Paulo processado com sucesso.
📊 (16/31) Extraindo: Velo Clube...
✅ Velo Clube processado com sucesso.
📊 (18/31) Extraindo: Athletico Paranaense...
✅ Athletico Paranaense processado com sucesso.
📊 (22/31) Extraindo: Coritiba...
✅ Coritiba processado com sucesso.
📊 (23/31) Extraind

## Insert no banco de dados MySQL

### map_campeonatos

In [19]:
map_campeonatos = {
    # Paulista
    'Palmeiras': 1,
    'Corinthians': 1,
    'Santos': 1,
    'São Paulo': 1,
    'Mirassol': 1,
    'Novorizontino': 1,
    'Botafogo-SP': 1,
    'RB Bragantino': 1,
    'Velo Clube': 1,
    'São Bernardo': 1,

    # Paranaense
    'Athletico Paranaense': 2,
    'Coritiba': 2,
    'Operario Ferroviario': 2,

    # Gaúcho
    'Internacional': 3,
    'Juventude': 3,

    # Mineiro
    'Atlético Mineiro': 4,
    'Cruzeiro': 4,
    'America-MG': 4,
    'Athletic Club': 4,
    'Villa Nova': 4,

    # Carioca
    'Flamengo': 5,
    'Fluminense': 5,
    'Vasco da Gama': 5,
    'Botafogo': 5,
    'Volta Redonda': 5
}

### limpar_dataframe_jogadores

In [ ]:
def limpar_dataframe_jogadores(df, map_campeonatos):
    """
    Limpa, normaliza e padroniza DataFrame de jogadores para carga no MySQL
    Realiza 8 etapas sequenciais de transformação de dados
    """

    # ====== ETAPA 1: Enriquecer com ID do campeonato ======
    # Mapeia cada time para seu respectivo campeonato via dicionário
    df['campeonato_id'] = df['time'].map(map_campeonatos)

    # Verifica se existem times que não foram mapeados (ausentes no dicionário)
    # Importante identificar para análise de dados incompletos
    faltantes = df[df['campeonato_id'].isnull()]['time'].unique()
    if len(faltantes) > 0:
        print("⚠️ Aviso: Times não encontrados no mapeamento de campeonatos:")
        print(faltantes)

    # ====== ETAPA 2: Remover registros inválidos ======
    # Remove linhas agregadas, vazias ou corrompidas do scraping
    # que não representam jogadores reais (ex: linhas de totalizadores do FBRef)
    df = df[df['nome'].notna()]  # Remove NULL
    df = df[df['nome'] != '0']   # Remove strings '0' (agregadores do FBRef)
    df = df[df['nome'] != 0]     # Remove inteiro 0 (em caso de conversão acidental)

    # ====== ETAPA 3: Normalizar valores numéricos inteiros ======
    # O FBRef usa vírgula como separador de milhar (formato brasileiro/europeu)
    # Converte para formato inteiro padrão esperado pelo MySQL
    colunas_int = [
        'idade', 'qt_jogos', 'qt_jogos_titular', 'minutos_jogados',
        'gols', 'assistencias', 'participacoes_gols',
        'gols_nao_penalti', 'gols_penalti', 'chutes_penalti',
        'cartoes_amarelo', 'cartoes_vermelho'
    ]

    for col in colunas_int:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(',', '', regex=False)  # Remove vírgula (separador de milhar)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')  # Converte para numérico (invalid → NaN)

    # ====== ETAPA 4: Normalizar valores numéricos decimais ======
    # Essas colunas já vêm no padrão correto (ponto como decimal)
    # Apenas converte o tipo para DECIMAL esperado pelo MySQL
    colunas_decimal = [
        'minutos_jogados_divid_90',
        'gols_90', 'assistencias_90', 'participacoes_gols_90',
        'gols_nao_penalti_90', 'participacoes_gols_e_penalti_90'
    ]

    for col in colunas_decimal:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # ====== ETAPA 5: Normalizar campos de texto ======
    # Padroniza strings removendo espaços extras e mantém formatos esperados
    if 'nacionalidade' in df.columns:
        # str[-3:]: Extrai últimas 3 caracteres
        # Mantém apenas código ISO-3 do país (ex: BRA, ESP, FRA)
        # Exemplo: "🇧🇷 Brazil" → "zil" (últimos 3 chars) → não ideal
        # Mas FBRef retorna como "BRA" (primeira linha), então funciona
        df['nacionalidade'] = (
            df['nacionalidade']
            .astype(str)
            .str[-3:]
        )

    # Remove espaços em branco nas extremidades (leading/trailing spaces)
    df['nome'] = df['nome'].astype(str).str.strip()
    df['time'] = df['time'].astype(str).str.strip()
    df['posicao'] = df['posicao'].astype(str).str.strip()

    # ====== ETAPA 6: Eliminar duplicatas ======
    # Remove registros repetidos que podem ter sido duplicados no scraping
    # (mesma linha extraída múltiplas vezes por alguma falha)
    df = df.drop_duplicates()

    # ====== ETAPA 7: Preparar para MySQL ======
    # MySQL não reconhece NaN do pandas - converte para None (NULL no banco)
    df = df.where(pd.notnull(df), None)

    # ====== ETAPA 8: Resetar índice ======
    # Restaura índice sequencial após todas as filtragens
    # Evita problemas de índices descontinuados ou fora de ordem
    df = df.reset_index(drop=True)

    # ====== RELATÓRIO DE EXECUÇÃO ======
    print(f"✅ Após limpeza: {len(df)} registros válidos prontos para inserção")

    # Debug útil para validar correção do tratamento de minutos
    if 'minutos_jogados' in df.columns:
        print(f"\nAmostras de minutos_jogados (primeiros 5 registros):")
        print(df[['minutos_jogados']].head())
        print(f"Tipo de dado: {df['minutos_jogados'].dtype}")

    return df

In [ ]:
# def limpar_dataframe_jogadores(df, map_campeonatos):
#     """
#     Limpa e padroniza DataFrame de jogadores para carga no MySQL
#     """

#     # CAMPEONATO_ID
#     df['campeonato_id'] = df['time'].map(map_campeonatos)

#     faltantes = df[df['campeonato_id'].isnull()]['time'].unique()
#     if len(faltantes) > 0:
#         print("⚠️ Times sem campeonato mapeado:")
#         print(faltantes)


#     # REMOVER LINHAS INVÁLIDAS
#     df = df[df['nome'].notna()]
#     df = df[df['nome'] != '0']
#     df = df[df['nome'] != 0]


#     # NUMÉRICOS
#     colunas_numericas = [
#         'idade', 'qt_jogos', 'qt_jogos_titular', 'minutos_jogados',
#         'minutos_jogados_divid_90', 'gols', 'assistencias',
#         'participacoes_gols', 'gols_nao_penalti', 'gols_penalti',
#         'chutes_penalti', 'cartoes_amarelo', 'cartoes_vermelho',
#         'gols_90', 'assistencias_90', 'participacoes_gols_90',
#         'gols_nao_penalti_90', 'participacoes_gols_e_penalti_90'
#     ]

#     for col in colunas_numericas:
#         if col in df.columns:
#             df[col] = (
#                 df[col]
#                 .astype(str)
#                 .str.replace('.', '', regex=False)   # remove milhar
#                 .str.replace(',', '.', regex=False)  # corrige decimal
#             )
#             df[col] = pd.to_numeric(df[col], errors='coerce')


#     # STRINGS
#     if 'nacionalidade' in df.columns:
#         df['nacionalidade'] = (
#             df['nacionalidade']
#             .astype(str)
#             .str[-3:]
#         )

#     df['nome'] = df['nome'].astype(str).str.strip()
#     df['time'] = df['time'].astype(str).str.strip()
#     df['posicao'] = df['posicao'].astype(str).str.strip()


#     # REMOVE DUPLICADOS
#     df = df.drop_duplicates()

#     # NULL PARA MYSQL
#     df = df.where(pd.notnull(df), None)

#     # FINAL
#     df = df.reset_index(drop=True)

#     print(f"✅ Após limpeza: {len(df)} registros válidos")

#     # debug útil
#     if 'minutos_jogados' in df.columns:
#         print(df[['minutos_jogados']].head())
#         print(df['minutos_jogados'].dtype)

#     return df

### tratar_valor_mysql

In [24]:
def tratar_valor_mysql(valor):
    if pd.isna(valor):
        return None
    return valor

### Insert

In [ ]:
import pandas as pd
import mysql.connector
import os


# ====== CARREGA DADOS BRUTOS ======
# Lê o CSV com estatísticas coletadas do FBRef
df = pd.read_csv('csv//stats_jogadores_2025.csv', sep=';')

# Limpa e padroniza dados antes da inserção
df = limpar_dataframe_jogadores(df, map_campeonatos)

# ====== CREDENCIAIS DO BANCO ======
# IMPORTANTE: Em produção, migrar para arquivo .env para segurança
DB_HOST     = "localhost"
DB_PORT     = 3310
DB_USER     = "admin"
DB_PASSWORD = "admin"
DB_DATABASE = "n8n"

try:
    # ====== CONECTAR AO BANCO DE DADOS ======
    # Estabelece conexão TCP/IP com MySQL
    print("🔗 Conectando ao banco de dados MySQL...")
    conn = mysql.connector.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_DATABASE 
    )

    cursor = conn.cursor()
    print("✅ Conexão estabelecida com sucesso!")

    # ====== PREPARAR TABELA ======
    # Remove tabela anterior se existir (evita dados obsoletos/inconsistentes)
    print("\n📋 Recriando tabela stats_jogadores_2025...")
    cursor.execute("DROP TABLE IF EXISTS stats_jogadores_2025")
    
    # Cria nova estrutura com tipos de dados otimizados
    # DECIMAL(10,2): Precision de 10 dígitos totais com 2 casas decimais
    # Exemplo: 12345678.90 (8 inteiros + 2 decimais = 10 totais)
    # Apropriado para estatísticas normalizadas por 90 minutos (ex: 2.45 gols/90)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS stats_jogadores_2025 (
        id                              INT AUTO_INCREMENT PRIMARY KEY,
        time                            VARCHAR(100) NOT NULL,
        campeonato_id                   INT NOT NULL,
        nome                            VARCHAR(150) NOT NULL,
        nacionalidade                   VARCHAR(50),
        posicao                         VARCHAR(100),
        idade                           INT,
        qt_jogos                        INT DEFAULT 0,
        qt_jogos_titular                INT DEFAULT 0,
        minutos_jogados                 INT DEFAULT 0,
        minutos_jogados_divid_90        DECIMAL(10,2),
        gols                            INT DEFAULT 0,
        assistencias                    INT DEFAULT 0,
        participacoes_gols              INT DEFAULT 0,
        gols_nao_penalti                INT DEFAULT 0,
        gols_penalti                    INT DEFAULT 0,
        chutes_penalti                  INT DEFAULT 0,
        cartoes_amarelo                 INT DEFAULT 0,
        cartoes_vermelho                INT DEFAULT 0,
        gols_90                         DECIMAL(10,2),
        assistencias_90                 DECIMAL(10,2),
        participacoes_gols_90           DECIMAL(10,2),
        gols_nao_penalti_90             DECIMAL(10,2),
        participacoes_gols_e_penalti_90 DECIMAL(10,2)
    );
    """)
    print("✅ Tabela criada com sucesso!")

    # ====== CONVERTER DataFrame PARA FORMATO MySQL ======
    # Substitui valores NaN do pandas por None (NULL no MySQL)
    df = df.where(pd.notnull(df), None)

    # ====== PREPARAR TUPLAS PARA INSERÇÃO ======
    # Converte linhas do DataFrame em tuplas, garantindo que NaN vire None
    # Seleciona colunas na mesma ordem da instrução INSERT
    dados = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in df[[
            'time','campeonato_id','nome','nacionalidade','posicao','idade',
            'qt_jogos','qt_jogos_titular','minutos_jogados',
            'minutos_jogados_divid_90','gols','assistencias',
            'participacoes_gols','gols_nao_penalti','gols_penalti',
            'chutes_penalti','cartoes_amarelo','cartoes_vermelho',
            'gols_90','assistencias_90','participacoes_gols_90',
            'gols_nao_penalti_90','participacoes_gols_e_penalti_90'
        ]].values
    ]

    # ====== EXECUTAR INSERÇÃO EM BATCH ======
    # Usa executemany para inserts múltiplos (mais eficiente que inserts individuais)
    # Reduz sobrecarga de rede e acelera a operação em 10-100x
    print(f"\n📝 Inserindo {len(dados)} registros no banco...")
    sql = """
    INSERT INTO stats_jogadores_2025 (
        time,
        campeonato_id,
        nome,
        nacionalidade,
        posicao,
        idade,
        qt_jogos,
        qt_jogos_titular,
        minutos_jogados,
        minutos_jogados_divid_90,
        gols,
        assistencias,
        participacoes_gols,
        gols_nao_penalti,
        gols_penalti,
        chutes_penalti,
        cartoes_amarelo,
        cartoes_vermelho,
        gols_90,
        assistencias_90,
        participacoes_gols_90,
        gols_nao_penalti_90,
        participacoes_gols_e_penalti_90
    )
    VALUES (
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s
    )
    """

    cursor.executemany(sql, dados)

    # ====== CONFIRMAR TRANSAÇÃO ======
    # Comita todas as alterações para persistência permanente no banco
    conn.commit()

    print(f"🚀 Sucesso! {cursor.rowcount} registros inseridos com sucesso!")

except mysql.connector.Error as err:
    print(f"❌ Erro de Banco de Dados: {err}")
    print(f"   Código do erro: {err.errno}")
    print(f"   SQL State: {err.sqlstate}")

finally:
    # ====== LIMPEZA DE RECURSOS ======
    # Fecha cursor e conexão em qualquer caso (sucesso ou erro)
    # Garante que não haja conexões pendentes com o banco
    cursor.close()
    conn.close()
    print("\n✅ Conexão fechada e recursos liberados.")

✅ Após limpeza: 1125 registros válidos
   minutos_jogados
0           3060.0
1           2717.0
2           2267.0
3           2319.0
4           2050.0
float64
🚀 Inseridos 1125 registros com sucesso!


# Fim